# Archived experiment — four-base-model Stacking (legacy)

**Historical experiment only.** This notebook combines Random Forest, XGBoost, Gaussian Process Regression, and LightGBM in the Ridge meta-model. Its saved results (e.g., R² = 0.8849) describe the **four-model** version, not the corrected three-model design. For the current three-model implementation, use `../05_stacking.ipynb` instead.

This file preserves the previously supplied experiment, including its archived outputs.

In [ ]:
# Project-relative paths (supports notebooks launched from the project root or notebooks/).
from pathlib import Path
import sys

_PROJECT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (p for p in _PROJECT_CANDIDATES if (p / "data" / "raw" / "mongo_test.test-result.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the project root or notebooks/ directory.")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "artifacts" / "models"
PREPROCESSING_DIR = PROJECT_ROOT / "artifacts" / "preprocessing"
SPLITS_DIR = PROJECT_ROOT / "artifacts" / "splits"
for _path in (DATA_PROCESSED, MODELS_DIR, PREPROCESSING_DIR, SPLITS_DIR):
    _path.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))


In [4]:
import numpy as np
import pandas as pd
import joblib
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from kafka_preprocessing import TargetScaler

In [5]:
rf_model   = joblib.load(str(MODELS_DIR / "rf_model.pkl"))
xgb_model  = joblib.load(str(MODELS_DIR / "xgb_model.pkl"))
gpr_model  = joblib.load(str(MODELS_DIR / "gpr_model.pkl"))
lgbm_model = joblib.load(str(MODELS_DIR / "lgbm_model.pkl"))

X_train_s     = joblib.load(str(SPLITS_DIR / "X_train_s.pkl"))
X_test_s      = joblib.load(str(SPLITS_DIR / "X_test_s.pkl"))
y_train       = joblib.load(str(SPLITS_DIR / "y_train.pkl"))
y_test        = joblib.load(str(SPLITS_DIR / "y_test.pkl"))
y_train_s     = joblib.load(str(SPLITS_DIR / "y_train_s.pkl"))
y_test_s      = joblib.load(str(SPLITS_DIR / "y_test_s.pkl"))
target_scaler = joblib.load(str(PREPROCESSING_DIR / "target_scaler.pkl"))
preprocessor  = joblib.load(str(PREPROCESSING_DIR / "preprocessor.pkl"))

print("✅ همه چیز لود شد!")

✅ همه چیز لود شد!


In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# RF نیاز به y دوبعدی داره
pred_rf_train = cross_val_predict(rf_model, X_train_s, y_train, cv=kf)
pred_rf_train = pred_rf_train.ravel()  # بعد از predict یک‌بعدی می‌کنیم

# XGBoost و LightGBM با y یک‌بعدی کار می‌کنن
pred_xgb_train  = cross_val_predict(xgb_model,  X_train_s, y_train.values.ravel(), cv=kf)
pred_lgbm_train = cross_val_predict(lgbm_model, X_train_s, y_train.values.ravel(), cv=kf)

# GPR روی scaled targets کار می‌کنه
pred_gpr_train_scaled = cross_val_predict(gpr_model, X_train_s, y_train_s.ravel(), cv=kf)
pred_gpr_train = target_scaler.scalers["throughput"].inverse_transform(
    pred_gpr_train_scaled.reshape(-1, 1)
).ravel()

# ترکیب پیش‌بینی‌ها
X_meta_train = np.column_stack([
    pred_rf_train,
    pred_xgb_train,
    pred_gpr_train,
    pred_lgbm_train,
])

print("✅ Meta-Features ساخته شدند!")
print(f"   شکل X_meta_train : {X_meta_train.shape}")

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Meta-Features ساخته شدند!
   شکل X_meta_train : (998, 4)


In [8]:
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta_train, y_train.values.ravel())

print("✅ Meta-Model آموزش دید!")
print(f"\n  وزن هر مدل:")
for name, coef in zip(["RF", "XGBoost", "GPR", "LightGBM"], meta_model.coef_):
    print(f"    {name:10} : {coef:.4f}")

✅ Meta-Model آموزش دید!

  وزن هر مدل:
    RF         : 0.5627
    XGBoost    : 0.6429
    GPR        : 0.1008
    LightGBM   : -0.2514


In [9]:

import numpy as np

corr = np.corrcoef(pred_xgb_train, pred_lgbm_train)[0, 1]
print(f"همبستگی XGBoost و LightGBM : {corr:.4f}")


همبستگی XGBoost و LightGBM : 0.9860


In [10]:
# پیش‌بینی روی test
pred_rf_test   = rf_model.predict(X_test_s)
pred_xgb_test  = xgb_model.predict(X_test_s)
pred_lgbm_test = lgbm_model.predict(X_test_s)
pred_gpr_test  = target_scaler.scalers["throughput"].inverse_transform(
    gpr_model.predict(X_test_s).reshape(-1, 1)
).ravel()

X_meta_test     = np.column_stack([pred_rf_test, pred_xgb_test, pred_gpr_test, pred_lgbm_test])
y_pred_stacking = meta_model.predict(X_meta_test)
true            = y_test.values[:, 0]

# مقایسه همه مدل‌ها
print("\n" + "="*55)
print("مقایسه نهایی همه مدل‌ها")
print("="*55)
print(f"\n{'مدل':<15} {'R²':>8} {'MAE':>12} {'RMSE':>12}")
print("-"*55)

models = {
    "Random Forest" : pred_rf_test,
    "XGBoost"       : pred_xgb_test,
    "GPR"           : pred_gpr_test,
    "LightGBM"      : pred_lgbm_test,
    "Stacking"      : y_pred_stacking,
}

for name, preds in models.items():
    r2   = r2_score(true, preds)
    mae  = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    print(f"{name:<15} {r2:>8.4f} {mae:>12.2f} {rmse:>12.2f}")


مقایسه نهایی همه مدل‌ها

مدل                   R²          MAE         RMSE
-------------------------------------------------------
Random Forest     0.8620      6666.97     10430.15
XGBoost           0.8824      6584.96      9627.00
GPR               0.8211      7565.13     11876.04
LightGBM          0.8556      7314.17     10667.87
Stacking          0.8849      6350.22      9527.09


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [12]:
BATCH_SIZE_VALUES  = [16384, 32768, 49152, 65536]
LINGER_MS_VALUES   = list(range(5, 506, 20))
COMPRESSION_VALUES = ["none", "snappy", "lz4", "gzip"]

def recommend_stacking(bill_size, top_n=3):
    combos = list(product(BATCH_SIZE_VALUES, LINGER_MS_VALUES, COMPRESSION_VALUES))
    grid   = pd.DataFrame(combos, columns=["batchSize", "lingerMs", "compressionType"])
    grid["billSize"] = bill_size

    X_grid = preprocessor.transform(grid)

    p_rf   = rf_model.predict(X_grid)
    p_xgb  = xgb_model.predict(X_grid)
    p_lgbm = lgbm_model.predict(X_grid)
    p_gpr  = target_scaler.scalers["throughput"].inverse_transform(
        gpr_model.predict(X_grid).reshape(-1, 1)
    ).ravel()

    X_meta_grid = np.column_stack([p_rf, p_xgb, p_gpr, p_lgbm])
    grid["predicted_throughput"] = meta_model.predict(X_meta_grid)
    grid = grid.sort_values("predicted_throughput", ascending=False)

    print(f"\n{'='*50}")
    print(f"Stacking  |  billSize = {bill_size:,}")
    print(f"{'='*50}")
    for rank, (_, row) in enumerate(grid.head(top_n).iterrows(), 1):
        print(f"\nRank #{rank}")
        print(f"  batchSize       : {int(row['batchSize'])}")
        print(f"  lingerMs        : {int(row['lingerMs'])} ms")
        print(f"  compressionType : {row['compressionType']}")
        print(f"  → throughput    : {row['predicted_throughput']:,.0f} docs/sec")



In [13]:
recommend_stacking(bill_size=309000)
recommend_stacking(bill_size=618000)
recommend_stacking(bill_size=927000)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



Stacking  |  billSize = 309,000

Rank #1
  batchSize       : 65536
  lingerMs        : 405 ms
  compressionType : snappy
  → throughput    : 181,765 docs/sec

Rank #2
  batchSize       : 65536
  lingerMs        : 325 ms
  compressionType : lz4
  → throughput    : 180,272 docs/sec

Rank #3
  batchSize       : 65536
  lingerMs        : 425 ms
  compressionType : snappy
  → throughput    : 180,262 docs/sec

Stacking  |  billSize = 618,000

Rank #1
  batchSize       : 65536
  lingerMs        : 445 ms
  compressionType : none
  → throughput    : 163,658 docs/sec

Rank #2
  batchSize       : 65536
  lingerMs        : 385 ms
  compressionType : none
  → throughput    : 163,580 docs/sec

Rank #3
  batchSize       : 65536
  lingerMs        : 405 ms
  compressionType : none
  → throughput    : 163,531 docs/sec

Stacking  |  billSize = 927,000

Rank #1
  batchSize       : 65536
  lingerMs        : 85 ms
  compressionType : none
  → throughput    : 150,617 docs/sec

Rank #2
  batchSize       : 65